# 4. Работа с файлами Parquet в pandas

Используя библиотеку pandas, напишите скрипт, который

- загружает файл Parquet с названием UNSW_NB15_training-set.parquet
- на его основе создает объект Dataframe
- выполняет фильтрацию по типу атаки (attack_cat) = 'Generic'
- создает сводную таблицу с подсчетом количества записей по протоколам (proto)
- сортирует по количеству записей по убыванию
- сохраняет полученный датафрейм в новый файл с именем UNSW_NB15_analyze.parquet

Источник данных - https://www.kaggle.com/datasets/dhoogla/unswnb15

Ссылка на задание: https://elearn.urfu.ru/mod/book/view.php?id=287538&chapterid=18497

In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("data/task4/UNSW_NB15_training-set.parquet")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 36 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   dur                175341 non-null  float32 
 1   proto              175341 non-null  category
 2   service            175341 non-null  category
 3   state              175341 non-null  category
 4   spkts              175341 non-null  int16   
 5   dpkts              175341 non-null  int16   
 6   sbytes             175341 non-null  int32   
 7   dbytes             175341 non-null  int32   
 8   rate               175341 non-null  float32 
 9   sload              175341 non-null  float32 
 10  dload              175341 non-null  float32 
 11  sloss              175341 non-null  int16   
 12  dloss              175341 non-null  int16   
 13  sinpkt             175341 non-null  float32 
 14  dinpkt             175341 non-null  float32 
 15  sjit               175341 non-null  float32 


In [4]:
df["attack_cat"].unique()  # Все уникальные значения в столбце "attack_cat"

['Normal', 'Backdoor', 'Analysis', 'Fuzzers', 'Shellcode', 'Reconnaissance', 'Exploits', 'DoS', 'Worms', 'Generic']
Categories (10, str): ['Analysis', 'Backdoor', 'DoS', 'Exploits', ..., 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']

In [5]:
df = df[df["attack_cat"] == "Generic"]

In [6]:
df

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
117204,0.000003,udp,dns,INT,2,0,114,0,333333.312500,152000000.0,...,0,0,22,10,0,0,0,0,Generic,1
117207,0.000009,udp,dns,INT,2,0,114,0,111111.109375,50666664.0,...,0,0,18,18,0,0,0,0,Generic,1
117208,0.000009,udp,dns,INT,2,0,114,0,111111.109375,50666664.0,...,0,0,17,17,0,0,0,0,Generic,1
117209,0.000012,udp,dns,INT,2,0,114,0,83333.328125,38000000.0,...,0,0,18,18,0,0,0,0,Generic,1
117211,0.000010,udp,dns,INT,2,0,114,0,100000.000000,45600000.0,...,0,0,8,8,0,0,0,0,Generic,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175335,0.000006,udp,dns,INT,2,0,114,0,166666.656250,76000000.0,...,0,0,33,17,0,0,0,0,Generic,1
175336,0.000009,udp,dns,INT,2,0,114,0,111111.109375,50666664.0,...,0,0,24,13,0,0,0,0,Generic,1
175338,0.000009,udp,dns,INT,2,0,114,0,111111.109375,50666664.0,...,0,0,3,3,0,0,0,0,Generic,1
175339,0.000009,udp,dns,INT,2,0,114,0,111111.109375,50666664.0,...,0,0,30,14,0,0,0,0,Generic,1


In [7]:
df["proto"].unique()

['udp', 'tcp', 'unas', 'sep', 'dgp', ..., 'scps', 'ipcomp', 'snp', 'pgm', 'srp']
Length: 122
Categories (133, str): ['3pc', 'a/n', 'aes-sp3-d', 'any', ..., 'xnet', 'xns-idp', 'xtp', 'zero']

In [9]:
pivot = df.pivot_table(index="proto", aggfunc="size")

In [ ]:
pivot = pd.DataFrame(pivot, columns=["count"])  # Создаём DataFrame из Series, чтобы добавить имя столбца и дальше сохранить файл в parquet

In [13]:
pivot.sort_values(by=["count"], ascending=False, inplace=True)

In [14]:
pivot

,count
proto,
udp,39229
tcp,486
unas,104
ospf,38
sctp,15
...,...
wsn,1
xnet,1
xns-idp,1


In [16]:
pivot.to_parquet("data/task4/UNSW_NB15_analyze.parquet")

---

Ниже код написанный в течениe учебного семестра (сейчас лето).

```python
import pandas as pd
import kagglehub

# Скачиваем датасет
# Download latest version
path = kagglehub.dataset_download("dhoogla/unswnb15")
print("Path to dataset files:", path)

# Загружаем Parquet в DataFrame
df = pd.read_parquet(f"{path}/UNSW_NB15_training-set.parquet")

# Фильтрация по типу атаки 'Generic'
# Оставляем строки, где значение "attack_cat" = "Generic"
df_filtered = df[df['attack_cat'] == 'Generic']

# Сводная таблица: количество записей по протоколам (proto)
# Группируем строки по "proto" (протиколам) и записываем в соседнем столбце количество строк того или иного протокола и задаём имя доп. столбцу
pivot = df_filtered.groupby('proto').size().reset_index(name='count')

# Сортировка по количеству по убыванию
pivot = pivot.sort_values('count', ascending=False)

# Сохраняем результат в Parquet
pivot.to_parquet('UNSW_NB15_analyze.parquet', index=False)
```